# GTEx model building with CLAMP (No FBM Version)

💡 **Environment:** `clamp-analyses`  

This notebook builds latent variable models from GTEx v8 RNA‑seq TPM data using CLAMP. It automates downloading and preprocessing the GTEx matrix, computes an SVD to estimate the model dimension, prepares pathway priors, runs CLAMP (base + full) and saves model outputs (B, Z, summaries) and intermediate files. Configuration and paths are controlled via `config.R`.

## Load libraries

In [2]:
# Create a timestamp to track the start of the analysis
start_time <- Sys.time()
cat("GTEx CLAMP and PLIER analysis started at:", format(start_time), "\n")

GTEx CLAMP and PLIER analysis started at: 2026-01-31 16:34:21 


In [3]:
if (!requireNamespace("PLIER", quietly = TRUE)) {
    devtools::install_github("wgmao/PLIER")
}

# Note: bigstatsr is no longer required for the main workflow
library(data.table)
library(dplyr)
library(rsvd)      # Using rsvd for SVD computation
library(glmnet)
library(Matrix)
library(knitr)
library(here)
library(PLIER)
library(CLAMP)

source(here("config.R"))

set.seed(config$GTEx$RANDOM_SVD_SEED)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:data.table’:

    between, first, last


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: Matrix

Loaded glmnet 4.1-10

here() starts at /home/msubirana/Documents/pivlab/clamp-analyses

Loading required package: RColorBrewer

Loading required package: gplots


---------------------
gplots 3.3.0 loaded:
  * Use citation('gplots') for citation info.
  * Homepage: https://talgalili.github.io/gplots/
  * Report issues: https://github.com/talgalili/gplots/issues
  * Ask questions: https://stackoverflow.com/questions/tagged/gplots
  * Suppress this message with: suppressPackageStartupMessages(library(gplots))
---------------------



Attaching package: ‘gplots’


The following object is masked from ‘package:stats’:

    lowess


Loading required package: pheatmap

Loadin

## Output directory

In [4]:
output_data_dir <- config$GTEx$OUTPUT_DIR
dir.create(output_data_dir, showWarnings = FALSE, recursive = TRUE)

output_data_dir

[1] "/home/msubirana/Documents/pivlab/clamp-analyses/output/gtex"

# Settings

In [5]:
block_size <- config$GENERAL$CHUNK_SIZE
N_CORES    <- config$GTEx$N_CORES

## Download GTEx 

In [6]:
url <- config$GTEx$URL
dest_dir <-  config$GTEx$DATASET_FOLDER
dest_gz  <- file.path(dest_dir, basename(url))

if (!file.exists(dest_gz)) {
  dir.create(dest_dir, recursive = TRUE, showWarnings = FALSE)
  download.file(url, dest_gz, mode = "wb")
  message("Downloaded to: ", dest_gz)
} else {
  message("File already exists, skipping download.")
}

File already exists, skipping download.



## Preprocess GTEx data (In-Memory Version)

In [ ]:
exprs_path  <- file.path(config$GTEx$DATASET_FOLDER, 'GTEx_Analysis_2017-06-05_v8_RNASeQCv1.1.9_gene_tpm.gct.gz')
output_file <- config$GTEx$DATASET_FILE

if (!file.exists(output_file)) {
  dir.create(dirname(output_file), recursive = TRUE, showWarnings = FALSE)
  exprs_data <- read.table(exprs_path, header = TRUE, sep = "\t", skip = 2, check.names = FALSE)
  saveRDS(exprs_data, config$GTEx$DATASET_FILE)
  message("File successfully written to: ", config$GTEx$DATASET_FILE)
} else {
  message("Output file already exists. Skipping.")
}

# Aggregate in-place by 'description'
gtex <- readRDS(here(config$GTEx$DATASET_FILE))
gtex <- as.data.table(gtex)
aggregated_gtex <- gtex[, lapply(.SD, sum), by = Description, .SDcols = is.numeric]

genes <- aggregated_gtex$Description
samples <- colnames(aggregated_gtex[, -1])
data_mat <- as.matrix(aggregated_gtex[, -1])
rownames(data_mat) <- genes

cat("Data matrix dimensions: ", dim(data_mat), "\n")

Output file already exists. Skipping.



Data matrix dimensions:  54592 17382 


In [9]:
gtex <- NULL

## Preprocess data

In [10]:
# Preprocess using CLAMP's in-memory function
prep_gtex <- preprocessCLAMP(
  Y = data_mat,
  mean_cutoff = config$GTEx$GENES_MEAN_CUTOFF,
  var_cutoff  = config$GTEx$GENES_VAR_CUTOFF
)

gtex_mat_filt <- prep_gtex$Y_filtered
gtex_rowStats <- prep_gtex$rowStats
gtex_genes <- rownames(gtex_mat_filt)

cat("Filtered matrix dimensions: ", dim(gtex_mat_filt), "\n")
cat("Number of genes after filtering: ", length(gtex_genes), "\n")

Filtered matrix dimensions:  23906 17382 
Number of genes after filtering:  23906 


In [11]:
data_mat <- NULL

In [12]:
# Z-score the filtered matrix
gtex_mat_zscore <- zscoreCLAMP(gtex_mat_filt, gtex_rowStats)

message("Z-score transformation complete")
cat("Z-scored matrix dimensions: ", dim(gtex_mat_zscore), "\n")

Z-score transformation complete



Z-scored matrix dimensions:  23906 17382 


In [13]:
gtex_mat_filt <- NULL

In [14]:
saveRDS(samples, file = file.path(output_data_dir, "gtex_samples_df.rds"))

In [15]:
saveRDS(gtex_genes, file = file.path(output_data_dir, "gtex_genes_df.rds"))

In [16]:
saveRDS(gtex_mat_zscore, file = file.path(output_data_dir, "gtex_zscore_df.rds"))

In [17]:
head(gtex_mat_zscore)

,GTEX-1117F-0226-SM-5GZZ7,GTEX-1117F-0426-SM-5EGHI,GTEX-1117F-0526-SM-5EGHJ,GTEX-1117F-0626-SM-5N9CS,GTEX-1117F-0726-SM-5GIEN,GTEX-1117F-1326-SM-5EGHH,GTEX-1117F-2426-SM-5EGGH,GTEX-1117F-2526-SM-5GZY6,GTEX-1117F-2826-SM-5GZXL,GTEX-1117F-2926-SM-5GZYI,⋯,GTEX-ZZPU-1126-SM-5N9CW,GTEX-ZZPU-1226-SM-5N9CK,GTEX-ZZPU-1326-SM-5GZWS,GTEX-ZZPU-1426-SM-5GZZ6,GTEX-ZZPU-1826-SM-5E43L,GTEX-ZZPU-2126-SM-5EGIU,GTEX-ZZPU-2226-SM-5EGIV,GTEX-ZZPU-2426-SM-5E44I,GTEX-ZZPU-2626-SM-5E45Y,GTEX-ZZPU-2726-SM-5NQ8O
WASH7P,1.5139414,-0.1112375,1.0449167,2.2783025,-0.2952013,0.3952429,2.5832515,4.2273240,1.9368923,2.7522992,⋯,-0.85869390,-0.63926325,0.3942485,-0.62534167,-0.57694760,-0.05754002,-0.75395057,-0.4440296,-1.1027856,-0.6727413
RP11-34P13.15,-0.2527007,-0.3213043,-0.2957023,-0.2840209,-0.3056420,-0.2718537,-0.2878607,-0.1725267,-0.2799266,-0.2157134,⋯,-0.24994808,-0.16271894,0.4705077,-0.14738271,-0.21589841,0.10317843,-0.12115151,-0.2495780,-0.3104186,-0.1814324
RP11-34P13.16,-0.2290288,-0.3321770,-0.3324351,-0.3106754,-0.3118786,-0.2826366,-0.3222012,-0.1543326,-0.2776653,-0.2462859,⋯,-0.20797205,-0.06203107,0.4016933,-0.07153038,-0.20724377,0.25223758,0.00272252,-0.2519063,-0.3248958,-0.1407170
RP11-34P13.14,-0.2911013,-0.2911013,-0.2911013,-0.2911013,-0.2911013,-0.2911013,-0.2911013,-0.2911013,0.0207658,-0.1096559,⋯,-0.29110127,-0.29110127,0.6469125,-0.11790180,-0.29110127,-0.29110127,0.17340459,-0.1903863,-0.2911013,0.1463984
RP11-34P13.18,0.6091633,-0.6029156,0.3535757,0.4464296,-0.9576751,-0.6469494,1.7100090,0.9863226,-0.1250529,0.7355213,⋯,-1.02506598,-0.72735903,-0.3119094,-0.93125479,-0.74631272,-0.18555151,-1.05837853,-0.7181694,-1.1631025,-0.5472032
AP006222.2,0.3230707,-0.5338201,-0.2776069,-0.3343933,-0.5886587,-0.6117329,-0.6144298,-0.6695681,-0.3805417,-0.6671708,⋯,-0.01854688,1.15898496,0.9776879,-0.24749060,-0.03607726,1.98456079,-0.36915441,-0.1495003,-0.2660698,3.3300546


## SVD computation using rsvd

Using the `rsvd` package for randomized SVD

In [18]:
if (!file.exists(file.path(output_data_dir, "gtex_svdRes_df.rds"))) {
  
  n_genes   <- nrow(gtex_mat_zscore)
  n_samples <- ncol(gtex_mat_zscore)
  SVD_K_gtex <- min(n_genes, n_samples) - 1
  
  message("Using SVD K = ", SVD_K_gtex)
  message("Computing SVD using rsvd package...")
  
  # Use rsvd for randomized SVD computation
  gtex_svdRes <- rsvd::rsvd(
    A = gtex_mat_zscore,
    k = SVD_K_gtex,
  )
  
  message("SVD computation complete")
  saveRDS(gtex_svdRes, file = file.path(output_data_dir, "gtex_svdRes_df.rds"))

} else {
  message("gtex_svdRes_df already exists, skipping SVD computation.")
}

Using SVD K = 17381

Computing SVD using rsvd package...



SVD computation complete



## Estimate K for CLAMP

In [19]:
CLAMP_K_gtex <- num.pc(list(d = gtex_svdRes$d)) * 2
message("Inferred CLAMP K = ", CLAMP_K_gtex)

Inferred CLAMP K = 76



In [20]:
saveRDS(CLAMP_K_gtex, file = file.path(output_data_dir, "CLAMP_K_gtex_df.rds"))

write.csv(
  as.data.frame(CLAMP_K_gtex),
  file = file.path(output_data_dir, "CLAMP_K_gtex_df.csv"),
  row.names = TRUE
)

## CLAMPbase

In [21]:
gtex_baseRes <- CLAMPbase(
  Y      = gtex_mat_zscore,
  svdres = gtex_svdRes,
  clamp_k = CLAMP_K_gtex,
  trace  = TRUE
)

****

CLAMP k is set to 76

L1 is set to 191.993746953477

L2 is set to 575.981240860431

Progress 1 / 200 | Bdiff=0.254485, minCor=0.773378

Progress 2 / 200 | Bdiff=0.029426, minCor=0.903755

Progress 3 / 200 | Bdiff=0.016988, minCor=0.957205

Progress 4 / 200 | Bdiff=0.012605, minCor=0.976182

Progress 5 / 200 | Bdiff=0.010108, minCor=0.986650

Progress 6 / 200 | Bdiff=0.008573, minCor=0.988342

Progress 7 / 200 | Bdiff=0.007697, minCor=0.988272

Progress 8 / 200 | Bdiff=0.007240, minCor=0.987677

Progress 9 / 200 | Bdiff=0.007019, minCor=0.987199

Progress 10 / 200 | Bdiff=0.006923, minCor=0.989553

Progress 11 / 200 | Bdiff=0.006851, minCor=0.990930

Progress 12 / 200 | Bdiff=0.006655, minCor=0.990990

Progress 13 / 200 | Bdiff=0.006198, minCor=0.989127

Progress 14 / 200 | Bdiff=0.005409, minCor=0.987897

Progress 15 / 200 | Bdiff=0.004424, minCor=0.989080

Progress 16 / 200 | Bdiff=0.003481, minCor=0.992104

Progress 17 / 200 | Bdiff=0.002695, minCor=0.994283

Progress 18 / 200 

In [22]:
gtex_baseRes$Z <- data.frame(gtex_baseRes$Z)
rownames(gtex_baseRes$Z) <- gtex_genes
head(gtex_baseRes$Z)

gtex_baseRes$B <- data.frame(gtex_baseRes$B)
colnames(gtex_baseRes$B) <- samples
head(gtex_baseRes$B)

,LV1,LV2,LV3,LV4,LV5,LV6,LV7,LV8,LV9,LV10,⋯,LV67,LV68,LV69,LV70,LV71,LV72,LV73,LV74,LV75,LV76
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
WASH7P,0.8272055,0.0000000,0.0000000,0.5193639,0,0.0000000,0,0,0,0,⋯,0,0,0,0,0.3144326,0,0,0,0,0.1091979
RP11-34P13.15,0.0000000,0.1797289,0.0000000,0.0000000,0,1.9481283,0,0,0,0,⋯,0,0,0,0,0.0000000,0,0,0,0,0.0000000
RP11-34P13.16,0.0000000,0.1740362,0.0000000,0.0000000,0,1.9668730,0,0,0,0,⋯,0,0,0,0,0.0000000,0,0,0,0,0.0000000
RP11-34P13.14,0.0000000,0.1732055,0.0000000,0.0000000,0,1.8990117,0,0,0,0,⋯,0,0,0,0,0.0000000,0,0,0,0,0.0000000
RP11-34P13.18,0.6988265,0.2158020,0.3752365,0.4051756,0,0.1724725,0,0,0,0,⋯,0,0,0,0,0.6025794,0,0,0,0,0.1719800
AP006222.2,0.0000000,0.0000000,0.0000000,0.0000000,0,1.5378608,0,0,0,0,⋯,0,0,0,0,0.0000000,0,0,0,0,0.1266051


,GTEX-1117F-0226-SM-5GZZ7,GTEX-1117F-0426-SM-5EGHI,GTEX-1117F-0526-SM-5EGHJ,GTEX-1117F-0626-SM-5N9CS,GTEX-1117F-0726-SM-5GIEN,GTEX-1117F-1326-SM-5EGHH,GTEX-1117F-2426-SM-5EGGH,GTEX-1117F-2526-SM-5GZY6,GTEX-1117F-2826-SM-5GZXL,GTEX-1117F-2926-SM-5GZYI,⋯,GTEX-ZZPU-1126-SM-5N9CW,GTEX-ZZPU-1226-SM-5N9CK,GTEX-ZZPU-1326-SM-5GZWS,GTEX-ZZPU-1426-SM-5GZZ6,GTEX-ZZPU-1826-SM-5E43L,GTEX-ZZPU-2126-SM-5EGIU,GTEX-ZZPU-2226-SM-5EGIV,GTEX-ZZPU-2426-SM-5E44I,GTEX-ZZPU-2626-SM-5E45Y,GTEX-ZZPU-2726-SM-5NQ8O
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
LV1,0.35404635,-0.43826198,0.405305198,0.593390138,-0.3093114,-0.10968317,0.945760803,0.41543240,0.17379170,0.03705816,⋯,-0.27054854,0.006169334,0.50261773,-0.19035956,0.21124090,0.856580517,-0.18235890,0.38836591,-0.32623475,0.32729002
LV2,-0.02031694,-0.03844093,-0.002866966,-0.041324036,-0.0697717,-0.05344384,-0.030983379,-0.04634647,-0.04860746,-0.05171169,⋯,-0.07229298,-0.074107590,-0.05159383,-0.07440363,-0.05759600,-0.065763276,-0.06897075,-0.05290481,-0.07123225,-0.05221654
LV3,-0.08577864,-0.10165204,-0.090274318,-0.089995347,-0.1019858,-0.09523285,-0.043631480,-0.06986274,-0.07971929,-0.09731560,⋯,-0.07947901,-0.080222890,-0.05917765,-0.08585270,-0.06969306,-0.001650053,-0.09475048,-0.10490047,-0.09162828,-0.11771457
LV4,-0.01848396,-0.14731210,-0.123239127,-0.117616727,-0.1085643,-0.07506894,-0.100940349,0.78055806,-0.05632713,0.99084234,⋯,-0.12097601,-0.116851714,-0.08752088,-0.09288025,-0.13986391,-0.111973913,0.28700577,-0.14967219,-0.13111210,-0.09635813
LV5,-0.03021653,-0.14102107,-0.028030926,-0.002938916,-0.2203093,-0.13593765,-0.145089615,0.12177560,0.02937177,-0.02970545,⋯,-0.08312276,0.048226111,-0.15538776,-0.06589111,-0.03361233,-0.066210945,0.26479788,-0.06994133,0.02932459,-0.02392196
LV6,0.05747919,-0.06922075,0.005747968,-0.004386508,-0.1131687,-0.06746142,-0.008063906,-0.04253440,-0.02869443,-0.04252615,⋯,-0.08411959,-0.047775999,-0.04657551,-0.09283512,-0.06295598,-0.097730345,-0.05538635,-0.03242057,-0.09692106,0.02249923


In [23]:
saveRDS(gtex_baseRes, file = file.path(output_data_dir, "CLAMPbase_df.rds"))

In [24]:
model_dir <- file.path(output_data_dir, "CLAMPbase_df")
dir.create(model_dir, showWarnings = FALSE, recursive = TRUE)

B <- gtex_baseRes$B
write.csv(B, file.path(model_dir, "B_df.csv"))

Z <- gtex_baseRes$Z
rownames(Z) <- gtex_genes
write.csv(Z, file.path(model_dir, "Z_df.csv"))

## Prepare pathway priors

In [26]:
gtex_gmtList <- list(
  BP = getGMT("https://maayanlab.cloud/Enrichr/geneSetLibrary?mode=text&libraryName=GO_Biological_Process_2025")
)

# prefix each gene‐set name with its library to guarantee uniqueness
for(lib in names(gtex_gmtList)) {
  names(gtex_gmtList[[lib]]) <- paste0(lib, "_", names(gtex_gmtList[[lib]]))
}

gtex_pathMat <- gmtListToSparseMat(gtex_gmtList)
gtex_matched <- getMatchedPathwayMat(gtex_pathMat, gtex_genes)
gtex_chatObj <- getChat(gtex_matched)

Auto-detected name: GO_Biological_Process_2025

Using cached file for GO_Biological_Process_2025

There are 12800 genes in the intersection between data and prior

Removing 1930 pathways

Inverting...

done



## CLAMPfull

In [27]:
gtex_fullRes <- CLAMPfull(
    Y = gtex_mat_zscore,
    priorMat = as.matrix(gtex_matched),
    clamp.base.result = gtex_baseRes,
    svdres = gtex_svdRes,
    clamp_k = CLAMP_K_gtex,
    doCrossval = TRUE,
    trace = TRUE
)

** CLAMPfull **

using provided CLAMPbase result

CLAMP k is set to 76

L1=191.993746953477; L2=575.981240860431

Progress 1 / 30 | Bdiff=0.000331

Progress 2 / 30 | Bdiff=0.001329

Progress 3 / 30 | Bdiff=0.000727

Estimated total runtime: ~2.0 min

Progress 4 / 30 | Bdiff=0.000649

Progress 5 / 30 | Bdiff=0.000737

Progress 6 / 30 | Bdiff=0.000554

Progress 7 / 30 | Bdiff=0.000490

Progress 8 / 30 | Bdiff=0.000508

Progress 9 / 30 | Bdiff=0.000534

Progress 10 / 30 | Bdiff=0.000471

Progress 11 / 30 | Bdiff=0.000603

Progress 12 / 30 | Bdiff=0.000450

Progress 13 / 30 | Bdiff=0.000459

Progress 14 / 30 | Bdiff=0.000445

Progress 15 / 30 | Bdiff=0.000557

Progress 16 / 30 | Bdiff=0.000409

Progress 17 / 30 | Bdiff=0.000533

Progress 18 / 30 | Bdiff=0.000533

Progress 19 / 30 | Bdiff=0.000610

Progress 20 / 30 | Bdiff=0.000490

Progress 21 / 30 | Bdiff=0.000700

Progress 22 / 30 | Bdiff=0.000314

Progress 23 / 30 | Bdiff=0.000505

Progress 24 / 30 | Bdiff=0.000496

Progress 25 / 30 | B

In [28]:
gtex_baseRes$Z <- data.frame(gtex_baseRes$Z)
rownames(gtex_baseRes$Z) <- gtex_genes
head(gtex_baseRes$Z)

gtex_baseRes$B <- data.frame(gtex_baseRes$B)
colnames(gtex_baseRes$B) <- samples
head(gtex_baseRes$B)

gtex_fullRes$summary <- gtex_fullRes$summary %>%
    dplyr::rename(LV = LV_index)  %>% 
    dplyr::mutate(LV = paste0('LV', LV))

,LV1,LV2,LV3,LV4,LV5,LV6,LV7,LV8,LV9,LV10,⋯,LV67,LV68,LV69,LV70,LV71,LV72,LV73,LV74,LV75,LV76
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
WASH7P,0.8272055,0.0000000,0.0000000,0.5193639,0,0.0000000,0,0,0,0,⋯,0,0,0,0,0.3144326,0,0,0,0,0.1091979
RP11-34P13.15,0.0000000,0.1797289,0.0000000,0.0000000,0,1.9481283,0,0,0,0,⋯,0,0,0,0,0.0000000,0,0,0,0,0.0000000
RP11-34P13.16,0.0000000,0.1740362,0.0000000,0.0000000,0,1.9668730,0,0,0,0,⋯,0,0,0,0,0.0000000,0,0,0,0,0.0000000
RP11-34P13.14,0.0000000,0.1732055,0.0000000,0.0000000,0,1.8990117,0,0,0,0,⋯,0,0,0,0,0.0000000,0,0,0,0,0.0000000
RP11-34P13.18,0.6988265,0.2158020,0.3752365,0.4051756,0,0.1724725,0,0,0,0,⋯,0,0,0,0,0.6025794,0,0,0,0,0.1719800
AP006222.2,0.0000000,0.0000000,0.0000000,0.0000000,0,1.5378608,0,0,0,0,⋯,0,0,0,0,0.0000000,0,0,0,0,0.1266051


,GTEX-1117F-0226-SM-5GZZ7,GTEX-1117F-0426-SM-5EGHI,GTEX-1117F-0526-SM-5EGHJ,GTEX-1117F-0626-SM-5N9CS,GTEX-1117F-0726-SM-5GIEN,GTEX-1117F-1326-SM-5EGHH,GTEX-1117F-2426-SM-5EGGH,GTEX-1117F-2526-SM-5GZY6,GTEX-1117F-2826-SM-5GZXL,GTEX-1117F-2926-SM-5GZYI,⋯,GTEX-ZZPU-1126-SM-5N9CW,GTEX-ZZPU-1226-SM-5N9CK,GTEX-ZZPU-1326-SM-5GZWS,GTEX-ZZPU-1426-SM-5GZZ6,GTEX-ZZPU-1826-SM-5E43L,GTEX-ZZPU-2126-SM-5EGIU,GTEX-ZZPU-2226-SM-5EGIV,GTEX-ZZPU-2426-SM-5E44I,GTEX-ZZPU-2626-SM-5E45Y,GTEX-ZZPU-2726-SM-5NQ8O
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
LV1,0.35404635,-0.43826198,0.405305198,0.593390138,-0.3093114,-0.10968317,0.945760803,0.41543240,0.17379170,0.03705816,⋯,-0.27054854,0.006169334,0.50261773,-0.19035956,0.21124090,0.856580517,-0.18235890,0.38836591,-0.32623475,0.32729002
LV2,-0.02031694,-0.03844093,-0.002866966,-0.041324036,-0.0697717,-0.05344384,-0.030983379,-0.04634647,-0.04860746,-0.05171169,⋯,-0.07229298,-0.074107590,-0.05159383,-0.07440363,-0.05759600,-0.065763276,-0.06897075,-0.05290481,-0.07123225,-0.05221654
LV3,-0.08577864,-0.10165204,-0.090274318,-0.089995347,-0.1019858,-0.09523285,-0.043631480,-0.06986274,-0.07971929,-0.09731560,⋯,-0.07947901,-0.080222890,-0.05917765,-0.08585270,-0.06969306,-0.001650053,-0.09475048,-0.10490047,-0.09162828,-0.11771457
LV4,-0.01848396,-0.14731210,-0.123239127,-0.117616727,-0.1085643,-0.07506894,-0.100940349,0.78055806,-0.05632713,0.99084234,⋯,-0.12097601,-0.116851714,-0.08752088,-0.09288025,-0.13986391,-0.111973913,0.28700577,-0.14967219,-0.13111210,-0.09635813
LV5,-0.03021653,-0.14102107,-0.028030926,-0.002938916,-0.2203093,-0.13593765,-0.145089615,0.12177560,0.02937177,-0.02970545,⋯,-0.08312276,0.048226111,-0.15538776,-0.06589111,-0.03361233,-0.066210945,0.26479788,-0.06994133,0.02932459,-0.02392196
LV6,0.05747919,-0.06922075,0.005747968,-0.004386508,-0.1131687,-0.06746142,-0.008063906,-0.04253440,-0.02869443,-0.04252615,⋯,-0.08411959,-0.047775999,-0.04657551,-0.09283512,-0.06295598,-0.097730345,-0.05538635,-0.03242057,-0.09692106,0.02249923


In [29]:
saveRDS(gtex_fullRes, file = file.path(output_data_dir, "CLAMPfull_df.rds"))

In [30]:
head(gtex_fullRes$B)
dim(gtex_fullRes$B)

,GTEX-1117F-0226-SM-5GZZ7,GTEX-1117F-0426-SM-5EGHI,GTEX-1117F-0526-SM-5EGHJ,GTEX-1117F-0626-SM-5N9CS,GTEX-1117F-0726-SM-5GIEN,GTEX-1117F-1326-SM-5EGHH,GTEX-1117F-2426-SM-5EGGH,GTEX-1117F-2526-SM-5GZY6,GTEX-1117F-2826-SM-5GZXL,GTEX-1117F-2926-SM-5GZYI,⋯,GTEX-ZZPU-1126-SM-5N9CW,GTEX-ZZPU-1226-SM-5N9CK,GTEX-ZZPU-1326-SM-5GZWS,GTEX-ZZPU-1426-SM-5GZZ6,GTEX-ZZPU-1826-SM-5E43L,GTEX-ZZPU-2126-SM-5EGIU,GTEX-ZZPU-2226-SM-5EGIV,GTEX-ZZPU-2426-SM-5E44I,GTEX-ZZPU-2626-SM-5E45Y,GTEX-ZZPU-2726-SM-5NQ8O
LV1,0.362795758,-0.32818029,0.61920877,0.54097376,-0.28743697,-0.108015801,0.78397505,0.27342950,0.11502039,0.02830698,⋯,-0.22509610,-0.04117791,0.32324875,-0.18001507,0.33701633,0.743953503,-0.19904732,0.66736218,-0.24971065,0.380828922
LV2,-0.016614699,-0.02558386,0.00241095,-0.04409215,-0.06336387,-0.047605167,-0.04308278,-0.04553768,-0.04470932,-0.04013797,⋯,-0.06986750,-0.07876094,-0.05495061,-0.07078349,-0.05382427,-0.097450355,-0.06803079,-0.04403892,-0.06132559,-0.041182584
LV3,-0.067517948,-0.09479451,-0.07678029,-0.08739586,-0.09324522,-0.076656098,-0.02855646,-0.05384298,-0.06797438,-0.07914963,⋯,-0.08340512,-0.08075884,-0.05876306,-0.09014377,-0.08401939,-0.012694445,-0.09777885,-0.11855088,-0.09585288,-0.112038974
LV4,0.009350565,-0.15870009,-0.08649113,-0.08793096,-0.08668544,-0.007594236,-0.06292614,0.94212764,-0.02055791,0.95844812,⋯,-0.12513564,-0.11886890,-0.06118977,-0.09299911,-0.16198875,-0.075090529,0.27500421,-0.15957917,-0.14135693,-0.087019129
LV5,-0.056097830,-0.23408159,-0.13242011,0.10032330,-0.27532673,-0.157821228,-0.04177080,0.17528056,0.06824517,-0.05152505,⋯,-0.10613078,0.09307817,-0.05858923,-0.08388920,-0.01905532,0.009885467,0.22787402,-0.08770850,0.01085056,-0.005085071
LV6,0.014830200,-0.08079260,-0.04708302,-0.01376257,-0.11573964,-0.127373191,-0.01996997,-0.07375275,-0.03248589,-0.07275384,⋯,-0.07691026,-0.03967860,-0.04382695,-0.08129020,-0.06397278,-0.087954532,-0.05195687,-0.03912318,-0.09480794,0.005716096


[1]    76 17382

In [31]:
model_dir <- file.path(output_data_dir, "CLAMPfull_df")
dir.create(model_dir, showWarnings = FALSE, recursive = TRUE)

B <- gtex_fullRes$B
colnames(B) <- samples
write.csv(B, file.path(model_dir, "B_df.csv"))

Z <- gtex_fullRes$Z
rownames(Z) <- gtex_genes
write.csv(Z, file.path(model_dir, "Z_df.csv"))

summary <- gtex_fullRes$summary
write.csv(summary, file.path(model_dir, "summary_df.csv"))